In [1]:
import sys
sys.path.append('..')

In [2]:
import matplotlib.pyplot as plt
import torch.nn as nn
from tqdm import tqdm
import hvplot
import hvplot.xarray

from forecast.evaluate import reduce_factor, spatial_acc_batch
from forecast.baselines import persistence_forecast, RidgeBaseline
from forecast.data import *
from forecast.model import *
from config import *

In [3]:
data_path = "../"+ config.SSTA_DAILY_PATH
landmask_path = "../"+ config.LANDMASK_PATH

In [90]:
ds_xr = xr.open_zarr(str(data_path))

In [44]:
horizon = max(config.LEAD_TIMES)
test_ds = SSTADataset(data_path, landmask_path, time_range=config.DL_TEST_RANGE, n_out=horizon)

In [45]:
loader = DataLoader(test_ds, batch_size=1, shuffle=False,  drop_last=True)
x_ld, y_ld = next(iter(loader))
print(x_ld.shape, y_ld.shape)

torch.Size([1, 14, 90, 180]) torch.Size([1, 28, 90, 180])


In [80]:
ckpt_path = '../forecast/checkpoints/conv_lstm_best.pt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
name = ckpt["model"]
n_in, n_out = ckpt["n_in"], ckpt["n_out"]
lstm_model = build_model(name, n_in=n_in, n_out=n_out)
lstm_model.load_state_dict(ckpt["state_dict"])
lstm_model.to('cpu').eval()
print(n_in, n_out)

14 1


In [47]:
out = lstm_model(x_ld)

### Forecasting

In [48]:
ocean = ~test_ds.land_mask   # (H, W) bool
ocean_t = torch.from_numpy(ocean)

In [49]:

methods = ["model", "persistence"]
accum = {
    m: {k: {"sse": 0.0, "n": 0, "acc_sum": 0.0, "acc_n": 0}
        for k in config.LEAD_TIMES}
    for m in methods
}

spatial_preds_model  = {k: [] for k in config.LEAD_TIMES}
spatial_preds_persistence  = {k: [] for k in config.LEAD_TIMES}
spatial_truths = {k: [] for k in config.LEAD_TIMES}


In [50]:
def pixel_acc_map(pred: np.ndarray, truth: np.ndarray, axis:int) -> np.ndarray:
    # pred, truth: (N, H, W)
    p = pred  - pred.mean(axis=axis, keepdims=True)
    t = truth - truth.mean(axis=axis, keepdims=True)
    num = (p * t).sum(axis=axis)
    den = np.sqrt((p**2).sum(axis=axis) * (t**2).sum(axis=axis)) + 1e-12
    return num / den   # (H, W)


In [51]:
from forecast.rollout import autoregressive_rollout

In [55]:
idx = 0
for x, y in tqdm(loader, desc="evaluating"):
    idx+=1
    preds_model = autoregressive_rollout(lstm_model, x, horizon, n_out=n_out)
    preds_persistence = persistence_forecast(x, horizon)

    for k in LEAD_TIMES:
        if k <= preds_model.shape[1]:            
            pred_m_k = preds_model[:, k - 1]
            pred_p_k = preds_persistence[:, k - 1]
            target_k = y[:, k - 1]

            # set all land at 0 insteak of masking to preserve shape
            pred_m_k[:, test_ds.land_mask] = 0
            pred_p_k[:, test_ds.land_mask] = 0
            target_k[:, test_ds.land_mask] = 0

            spatial_preds_model[k].append(pred_m_k)   # (B, H, W)
            spatial_preds_persistence[k].append(pred_p_k)
            spatial_truths[k].append(target_k)

    if idx == 10:
        break


evaluating:   1%|          | 9/1785 [03:42<12:10:30, 24.68s/it]


In [84]:
def pixel_rmse_map(pred: np.ndarray, truth: np.ndarray) -> np.ndarray:
     # pred, truth: (N, H, W)
     return np.sqrt(np.mean((pred - truth) ** 2, axis=0))  # (H, W)

In [85]:
acc_model_worldmap = []
acc_persistence_worldmap = []
rmse_model_worldmap = []
rmse_persistence_worldmap = []

for k in LEAD_TIMES:
    a = accum[m][k]
    # rmse = (a["sse"] / max(a["n"], 1)) ** 0.5

    all_preds_model  = np.concatenate(spatial_preds_model[k],  axis=0)  # (N, H, W)
    all_preds_persistence  = np.concatenate(spatial_preds_persistence[k],  axis=0)  # (N, H, W)
    all_truth = np.concatenate(spatial_truths[k], axis=0)
    

    # ACC
    acc_model = pixel_acc_map(all_preds_model, all_truth, axis=0)            # (H, W)
    acc_model[test_ds.land_mask] = np.nan                             # mask land
    acc_model_worldmap.append(acc_model)

    acc_persistence = pixel_acc_map(all_preds_persistence, all_truth, axis=0)            # (H, W)
    acc_persistence[test_ds.land_mask] = np.nan                             # mask land
    acc_persistence_worldmap.append(acc_persistence)


    # RMSE
    rmse_model = pixel_rmse_map(all_preds_model, all_truth)
    rmse_model[test_ds.land_mask] = np.nan
    rmse_model_worldmap.append(rmse_model)

    rmse_persistence = pixel_rmse_map(all_preds_persistence, all_truth)
    rmse_persistence[test_ds.land_mask] = np.nan
    rmse_persistence_worldmap.append(rmse_persistence)

In [86]:
acc_model_worldmap = np.array(acc_model_worldmap)
acc_persistence_worldmap = np.array(acc_persistence_worldmap)

rmse_model_worldmap = np.array(rmse_model_worldmap)
rmse_persistence_worldmap = np.array(rmse_persistence_worldmap)

In [87]:
acc_model_worldmap.shape, acc_persistence_worldmap.shape

((11, 90, 180), (11, 90, 180))

In [91]:
ds_xr = ds_xr.coarsen(lat=2, lon=2).mean()
ds_xr.ssta.shape

(15566, 90, 180)

In [113]:
rmse_max = max(np.nanquantile(rmse_model_worldmap, 0.99), np.nanquantile(rmse_persistence_worldmap, 0.99))

In [114]:
np.array([0,rmse_max])

array([0.        , 1.32641983])

In [ ]:
ds = xr.Dataset(
    data_vars=dict(
        model_acc=(['lead_time', 'lat', 'lon'], acc_model_worldmap),
        persistence_acc=(['lead_time', 'lat', 'lon'], acc_persistence_worldmap),
        model_rmse=(['lead_time', 'lat', 'lon'], rmse_model_worldmap),
        persistence_rmse=(['lead_time', 'lat', 'lon'], rmse_persistence_worldmap),
        rmse_range = np.array([0,rmse_max]),

    ),
    coords = dict(
        lead_time=np.array(LEAD_TIMES),
        lat=("lat", ds_xr.ssta.lat.values),
        lon=("lon", ds_xr.ssta.lon.values),
    ),
    attrs=dict(
        acc_metric='Anomaly Correlation Coefficient',
        rmse_metric='Root Mean Square Error',
        model=name,
        input_window=n_in, )
)

ds

SyntaxError: invalid syntax (1995066960.py, line 20)

In [82]:
ds.to_zarr(f'{ds.model}_n_in{ds.input_window}.zarr')


/home/smaug/miniconda3/envs/mhw-detection/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [115]:
# da = xr.DataArray(acc_map, dims=["lat", "lon"])
ds['model_rmse'].hvplot(x='lon', y='lat', cmap="RdYlBu_r", clim=(ds.rmse_range[0],1.32), title=f'Anomaly Correlation Coefficient at different Lead Times using {ds.model}')

BokehModel(combine_events=True, render_bundle={'docs_json': {'335d589e-1a22-47e2-aaed-c1592dffd376': {'version…

In [117]:
ds['persistence_rmse'].hvplot(x='lon', y='lat', cmap="RdYlBu_r", clim=(0, 1.32), title=f'Anomaly Correlation Coefficient at different Lead Times using {ds.model}')

BokehModel(combine_events=True, render_bundle={'docs_json': {'411e15d8-8d58-47a8-bc2b-f277fdfa93da': {'version…